# Análisis de un dataset académico — unidad de análisis **estudiante-semestre**

**Maestría en Inteligencia Artificial — UIDE**
Semana 1: Aprendizaje Automático Estadístico

---

## Objetivo y restricción

Ejecutar el análisis estadístico completo —definición de la unidad de análisis, curaduría, control de
*data leakage*, clasificación y codificación de variables, exploración gráfica y análisis matricial— sobre el
**dataset original y únicamente sobre él**. No se simula actividad semanal, ni asignaturas, ni variables
adicionales: se trabaja con las 100 filas y 9 columnas tal como vienen.

Esa restricción es deliberada y condiciona los resultados. Conviene anticipar tres consecuencias, porque
determinan lo que el análisis puede y no puede concluir:

1. **La unidad de análisis queda determinada por los datos, no elegida.** Sin columna de asignatura ni de
   período, la única unidad construible es *un estudiante en el único semestre observado*.
2. **La ventana temporal ≤ Semana 8 tiene poco que filtrar.** El dataset contiene una sola variable
   genuinamente futura, así que el detector de fuga eliminará una columna en lugar de nueve. Se incluye una
   **prueba explícita del detector** para demostrar que las reglas que no se activan funcionan igualmente.
3. **Las variables son estadísticamente independientes entre sí por construcción.** El generador las extrae con
   llamadas separadas a `np.random`, sin ninguna estructura de dependencia. El análisis exploratorio debería,
   por tanto, **no encontrar señal**. Eso no es un fallo del método: es el resultado correcto, y saber
   reconocerlo es tan importante como saber detectar un patrón.

## Contenido

| Sección | Contenido |
|---|---|
| **1** | Unidad de análisis e ingesta — agregación en Pandas |
| **2** | Curaduría y control de *data leakage* con ventana temporal ≤ Semana 8 |
| **3** | Clasificación de 8 variables y estrategia de *encoding* |
| **4** | Visualización exploratoria: 5 gráficos |
| **5** | Análisis matricial en NumPy: esperanza, varianza, covarianza y heterocedasticidad entre sedes |
| **6** | Conclusiones |

---
## 0. El dataset original

La celda siguiente es el dataset entregado, sin modificación alguna: **100 filas × 9 columnas**, una fila por
registro de estudiante. Incluye por construcción cinco defectos de calidad:

| Defecto | Columna | Tratamiento (Sección 2) |
|---|---|---|
| 6 `id_estudiante` repetidos | `id_estudiante` | agregación por clave |
| Categoría inconsistente `quito_2` | `sede` | normalización con diccionario |
| ~1 de cada 6 valores ausentes | `nivel_satisfaccion` | imputación por moda |
| Fecha almacenada como texto `%Y/%m/%d` | `fecha_ultimo_acceso` | `pd.to_datetime` + censura a la ventana |
| Cola exponencial (outliers) | `tiempo_conexion_min` | winsorización por IQR |

In [ ]:
# Verifica el intérprete e instala las dependencias en el kernel realmente activo.
import importlib.util
import subprocess
import sys

dependencias = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "seaborn": "seaborn"}
faltantes = [paquete for modulo, paquete in dependencias.items() if importlib.util.find_spec(modulo) is None]
print(f"Intérprete activo: {sys.executable}")
if faltantes:
    print(f"Instalando dependencias faltantes: {', '.join(faltantes)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilos visuales
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

# 1. Carga del Dataset Simulado
np.random.seed(42)
n = 100

data = {
    "id_estudiante": [f"EST-{i:03d}" for i in range(1, 95)] + ["EST-010", "EST-012", "EST-025", "EST-030", "EST-050", "EST-001"],
    "sede": np.random.choice(["Quito", "Guayaquil", "Cuenca", "quito_2"], n),
    "modalidad": np.random.choice(["Presencial", "En línea"], n),
    "promedio_acumulado": np.random.uniform(5.0, 10.0, n),
    "tiempo_conexion_min": np.random.exponential(scale=120, size=n),
    "entregas_semana_1_8": np.random.randint(0, 10, n),
    "nivel_satisfaccion": np.random.choice([1, 2, 3, 4, 5, np.nan], n),
    "calificacion_final_semestre": np.random.uniform(0, 10, n),  
    "fecha_ultimo_acceso": pd.date_range(start="2026-01-15", periods=n, freq="D").strftime("%Y/%m/%d")
}

df_raw = pd.DataFrame(data)
print(f"Dimensiones iniciales del dataset: {df_raw.shape}")
df_raw.head()

---
## Calidad de datos con el framework DAMA

Se evalúan los campos solicitados antes de realizar cualquier limpieza o consolidación:

- **`id_estudiante` — Unicidad:** cada identificador debe aparecer una sola vez.  \n  $KPI = (\text{IDs únicos} / \text{total de registros}) \times 100$
- **`promedio_acumulado` — Integridad:** el campo debe estar informado (no nulo).  \n  $KPI = (\text{valores no nulos} / \text{total de registros}) \times 100$
- **`promedio_acumulado` — Consistencia:** debe ser numérico y pertenecer a la escala académica de 0 a 10.  \n  $KPI = (\text{valores numéricos entre 0 y 10} / \text{total de registros}) \times 100$


In [ ]:
# Reglas y KPIs de calidad de datos basados en DAMA
TOTAL_REGISTROS = len(df_raw)

# 1. id_estudiante - dimensión de unicidad
ids_informados = df_raw["id_estudiante"].notna() & df_raw["id_estudiante"].astype("string").str.strip().ne("")
cantidad_ids_unicos = df_raw.loc[ids_informados, "id_estudiante"].nunique()
kpi_unicidad_id = cantidad_ids_unicos / TOTAL_REGISTROS * 100

# 2. promedio_acumulado - dimensiones de integridad y consistencia
promedio_numerico = pd.to_numeric(df_raw["promedio_acumulado"], errors="coerce")
promedios_no_nulos = df_raw["promedio_acumulado"].notna().sum()
promedios_consistentes = (promedio_numerico.notna() & promedio_numerico.between(0, 10, inclusive="both")).sum()
kpi_integridad_promedio = promedios_no_nulos / TOTAL_REGISTROS * 100
kpi_consistencia_promedio = promedios_consistentes / TOTAL_REGISTROS * 100

reglas_dama = pd.DataFrame([
    {
        "Dataset": "Dataset_estudiante_semestre",
        "Campo": "id_estudiante",
        "Característica DAMA": "Unicidad",
        "Regla o criterio aplicado": "Cada ID debe ser único y no repetirse",
        "KPI / Métrica de calidad": "% de IDs únicos",
        "Fórmula": "(IDs únicos / total de registros) × 100",
    },
    {
        "Dataset": "Dataset_estudiante_semestre",
        "Campo": "promedio_acumulado",
        "Característica DAMA": "Integridad",
        "Regla o criterio aplicado": "El promedio debe tener un valor no nulo",
        "KPI / Métrica de calidad": "% de promedios informados",
        "Fórmula": "(Valores no nulos / total de registros) × 100",
    },
    {
        "Dataset": "Dataset_estudiante_semestre",
        "Campo": "promedio_acumulado",
        "Característica DAMA": "Consistencia",
        "Regla o criterio aplicado": "El promedio debe ser numérico y estar entre 0 y 10",
        "KPI / Métrica de calidad": "% de promedios consistentes",
        "Fórmula": "(Valores válidos entre 0 y 10 / total de registros) × 100",
    },
])

resultados_kpi_dama = pd.DataFrame([
    {"Campo": "id_estudiante", "Dimensión DAMA": "Unicidad",
     "Numerador": cantidad_ids_unicos, "Denominador": TOTAL_REGISTROS, "KPI (%)": kpi_unicidad_id},
    {"Campo": "promedio_acumulado", "Dimensión DAMA": "Integridad",
     "Numerador": promedios_no_nulos, "Denominador": TOTAL_REGISTROS, "KPI (%)": kpi_integridad_promedio},
    {"Campo": "promedio_acumulado", "Dimensión DAMA": "Consistencia",
     "Numerador": promedios_consistentes, "Denominador": TOTAL_REGISTROS, "KPI (%)": kpi_consistencia_promedio},
])
resultados_kpi_dama["Estado (meta 100%)"] = np.where(
    resultados_kpi_dama["KPI (%)"].eq(100), "CUMPLE", "NO CUMPLE"
)
resultados_kpi_dama["KPI (%)"] = resultados_kpi_dama["KPI (%)"].round(2)

print("REGLAS DE CALIDAD DE DATOS (DAMA)")
display(reglas_dama)
print("RESULTADOS DE LOS KPI")
display(resultados_kpi_dama)

# Evidencia de los registros que incumplen la unicidad
ids_duplicados = (df_raw[df_raw.duplicated(subset=["id_estudiante"], keep=False)]
                  .sort_values("id_estudiante"))
print(f"IDs únicos: {cantidad_ids_unicos} de {TOTAL_REGISTROS} -> KPI de unicidad: {kpi_unicidad_id:.2f}%")
print(f"Filas asociadas a IDs repetidos: {len(ids_duplicados)}")
display(ids_duplicados[["id_estudiante", "promedio_acumulado"]])


---
# 1. Unidad de análisis e ingesta

## 1.1 La unidad la determinan los datos

La unidad de análisis define **qué representa una fila** y, por tanto, qué significaría una predicción. El
enunciado plantea dos candidatas; el dataset solo admite una:

| Unidad candidata | ¿Construible aquí? | Motivo |
|---|---|---|
| estudiante-asignatura | **No** | no existe columna `asignatura`, ni ninguna otra que identifique la materia. Construirla exigiría inventar datos |
| **estudiante-semestre** | **Sí** | es el grano natural del dataset: cada fila describe a un estudiante en un período |

Queda una precisión necesaria. No hay columna `semestre`, así que hay que verificar que los datos cubren **un
único período** antes de afirmar que la clave `id_estudiante` identifica la unidad. La celda siguiente lo
comprueba con las fechas: si todas caen dentro de una misma ventana de ~3 meses, se trata de un solo semestre y
entonces

$$\text{unidad estudiante-semestre} \;\equiv\; \text{una fila por } \texttt{id\_estudiante}$$

Con esa verificación hecha, la clave primaria es `id_estudiante` y las 6 repeticiones dejan de ser un detalle
de limpieza para convertirse en el **problema central de la ingesta**: son registros conflictivos del mismo
estudiante que hay que consolidar en una sola fila.

In [ ]:
import re

# ---------------------------------------------------------------- Estructura del dataset original
print(f"Dimensiones: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas\n")
print(f"{'Columna':<30} | {'Tipo':<12} | {'Únicos':>7} | {'Nulos':>6} | Ejemplo")
print("-" * 96)
for c in df_raw.columns:
    print(f"{c:<30} | {str(df_raw[c].dtype):<12} | {df_raw[c].nunique():>7} | "
          f"{int(df_raw[c].isna().sum()):>6} | {repr(df_raw[c].iloc[0])[:28]}")

# ---------------------------------------------------------------- ¿Existe un eje de asignatura o de semana?
ejes_ausentes = [e for e in ["asignatura", "materia", "curso", "semestre", "periodo", "semana"]
                 if not any(e in c.lower() for c in df_raw.columns)]
print(f"\nEjes de desagregación ausentes en el dataset: {ejes_ausentes}")
print("-> la unidad 'estudiante-asignatura' NO es construible sin inventar datos.")

# ---------------------------------------------------------------- ¿Un solo período?
fechas = pd.to_datetime(df_raw["fecha_ultimo_acceso"], format="%Y/%m/%d")
FECHA_INICIO = fechas.min()
rango_dias = (fechas.max() - FECHA_INICIO).days
print(f"\nRango de fechas: {FECHA_INICIO.date()} a {fechas.max().date()}  ({rango_dias} días ~ "
      f"{rango_dias / 7:.1f} semanas)")
assert rango_dias <= 200, "El rango excede un semestre: habría que separar por período"
print("-> todas las observaciones caen en un único semestre; la clave de la unidad es 'id_estudiante'.")

# ---------------------------------------------------------------- Ventana temporal permitida
SEMANA_CORTE = 8
FECHA_CORTE  = FECHA_INICIO + pd.Timedelta(weeks=SEMANA_CORTE)
CLAVE = ["id_estudiante"]
print(f"\nVentana de observación permitida: {FECHA_INICIO.date()} -> {FECHA_CORTE.date()} "
      f"(semana {SEMANA_CORTE})")

# ---------------------------------------------------------------- Los duplicados son registros CONFLICTIVOS
dups = df_raw[df_raw.duplicated(subset=CLAVE, keep=False)].sort_values("id_estudiante")
print(f"\nRegistros con id_estudiante repetido: {len(dups)} filas / {dups['id_estudiante'].nunique()} estudiantes")
print("No son copias exactas: cada registro trae valores distintos, así que hay que consolidarlos.\n")
print(dups[["id_estudiante", "sede", "modalidad", "promedio_acumulado",
            "entregas_semana_1_8", "nivel_satisfaccion"]].round(2).to_string(index=False))

## 1.2 Agregación a la unidad estudiante-semestre

Consolidar los registros repetidos exige decidir **una regla de agregación por columna**, y cada regla es un
supuesto explícito sobre qué significa el conflicto:

| Columna | Agregación | Supuesto |
|---|---|---|
| `promedio_acumulado`, `tiempo_conexion_min` | **media** | los dos registros son mediciones parciales del mismo estudiante; promediarlas es el estimador natural |
| `entregas_semana_1_8` | **suma** | es un conteo acumulado: dos registros parciales de entregas se suman |
| `nivel_satisfaccion` | **máximo** | escala ordinal: la media daría valores como 3.5, que no existen en la escala |
| `sede`, `modalidad` | **primero** | atributos que deberían ser constantes; se conserva el primer registro y se **audita** cuántos entraban en conflicto |
| `fecha_ultimo_acceso` | **máximo** | el último acceso del estudiante es el más reciente de sus registros |
| `calificacion_final_semestre` | **media** | igual criterio que las continuas |

La alternativa —`drop_duplicates`, descartar el segundo registro— tiraría información sin justificación. Al
agregar se conserva todo y, además, se genera `n_registros`, que documenta cuántas filas originales alimentaron
cada estudiante. Se audita explícitamente si alguno de los duplicados traía **sede o modalidad contradictorias**,
porque en ese caso el `first()` está tomando una decisión arbitraria que el analista debe conocer.

In [ ]:
# ---------------------------------------------------------------- Auditoría previa: ¿hay conflictos reales?
conflictos = (df_raw.groupby("id_estudiante")[["sede", "modalidad"]].nunique()
                    .query("sede > 1 or modalidad > 1"))
print(f"Estudiantes con 'sede' o 'modalidad' contradictoria entre sus registros: {len(conflictos)}")
if len(conflictos):
    print(conflictos.to_string())
    print("-> el criterio first() resuelve el empate de forma arbitraria; queda documentado.")

# ---------------------------------------------------------------- Agregación a estudiante-semestre
AGG_SEMESTRE = {
    "n_registros":                 ("sede", "size"),
    "sede":                        ("sede", "first"),
    "modalidad":                   ("modalidad", "first"),
    "promedio_acumulado":          ("promedio_acumulado", "mean"),
    "tiempo_conexion_min":         ("tiempo_conexion_min", "mean"),
    "entregas_semana_1_8":         ("entregas_semana_1_8", "sum"),
    "nivel_satisfaccion":          ("nivel_satisfaccion", "max"),
    "calificacion_final_semestre": ("calificacion_final_semestre", "mean"),
    "fecha_ultimo_acceso":         ("fecha_ultimo_acceso", "max"),
}

df_semestre = df_raw.groupby("id_estudiante", as_index=False).agg(**AGG_SEMESTRE)

print(f"\n{'Etapa':<44} | {'Filas':>7} | {'Columnas':>9}")
print("-" * 68)
print(f"{'df_raw (registros crudos)':<44} | {df_raw.shape[0]:>7} | {df_raw.shape[1]:>9}")
print(f"{'df_semestre (unidad estudiante-semestre)':<44} | {df_semestre.shape[0]:>7} | {df_semestre.shape[1]:>9}")
print(f"\nEstudiantes con más de un registro consolidado: {(df_semestre['n_registros'] > 1).sum()}")
print(f"Clave 'id_estudiante' única: {not df_semestre['id_estudiante'].duplicated().any()}")
assert not df_semestre["id_estudiante"].duplicated().any(), "La clave de la unidad no es única"

df_semestre.head()

---
# 2. Curaduría y control de *data leakage*

## 2.1 Diagnóstico

Antes de corregir hay que medir. El reporte cuantifica duplicados, nulos y tipos efectivos; se vuelve a
ejecutar tras la curaduría para que la mejora sea verificable y no una afirmación.

In [ ]:
def reporte_calidad(df, titulo, clave=CLAVE):
    """Diagnóstico de calidad: dimensiones, duplicados, nulos y tipos."""
    print(f"{'=' * 88}\n{titulo}\n{'=' * 88}")
    print(f"Dimensiones                : {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"Filas duplicadas exactas   : {df.duplicated().sum()}")
    print(f"Duplicados por clave       : {df.duplicated(subset=clave).sum()}")
    print(f"Celdas nulas (total)       : {int(df.isna().sum().sum())}")
    print(f"\n{'Columna':<30} | {'Tipo':<14} | {'Nulos':>6} | {'% nulos':>8} | {'Únicos':>7}")
    print("-" * 88)
    for c in df.columns:
        nulos = int(df[c].isna().sum())
        print(f"{c:<30} | {str(df[c].dtype):<14} | {nulos:>6} | {100 * nulos / len(df):>7.1f}% | {df[c].nunique():>7}")
    print()

reporte_calidad(df_semestre, "DIAGNÓSTICO ANTES DE LA CURADURÍA")
print("Categorías crudas de 'sede':", sorted(df_semestre["sede"].unique()))
print("Ejemplo de 'fecha_ultimo_acceso':", repr(df_semestre["fecha_ultimo_acceso"].iloc[0]),
      "-> texto, no fecha")
print(f"Asimetría de 'tiempo_conexion_min': {df_semestre['tiempo_conexion_min'].skew():.2f} -> cola derecha")

## 2.2 Curaduría programática

Cada tratamiento se aplica con una regla explícita y queda registrado en una **bitácora** con el número de
filas o valores afectados:

| Problema | Tratamiento | Por qué |
|---|---|---|
| `sede`: `quito_2` | minúsculas + diccionario `MAPA_SEDES` | una sede partida en dos categorías fragmenta los grupos y sesga las comparaciones de la Sección 5 |
| Fechas en texto | `pd.to_datetime` + `clip(upper=FECHA_CORTE)` | sin tipo fecha no hay aritmética temporal; el `clip` **censura** el último acceso a la ventana observable, porque en la semana 8 el futuro aún no ha ocurrido |
| Nulos en `nivel_satisfaccion` | moda | variable ordinal discreta: la media daría valores como 3.4, inexistentes en la escala |
| Outliers en `tiempo_conexion_min` | **winsorización** por IQR, acotada a ≥ 0 | la cola exponencial es real, no un error; se acota su influencia sin borrar al estudiante. El piso se fuerza a 0 porque el tiempo de conexión no puede ser negativo |

Los duplicados no aparecen en esta tabla porque **ya se resolvieron en la Sección 1** mediante agregación, que
es donde correspondía: son un problema de definición de la unidad de análisis, no de limpieza.

Se deriva además `dias_hasta_ultimo_acceso`, la distancia en días entre el inicio del semestre y el último
acceso censurado, que convierte una fecha —inutilizable como predictor -- en una magnitud numérica.

> **Supuesto documentado.** `nivel_satisfaccion` se asume recogido en una encuesta **dentro** de la ventana
> (semana 4). Si se levantara al cierre del semestre sería fuga de información y habría que excluirlo. Es una
> decisión de negocio, no estadística, y por eso queda escrita.

In [ ]:
df_curado = df_semestre.copy()
bitacora  = []

# ------------------------------------------------------------ a) Normalización de categorías
MAPA_SEDES  = {"quito": "Quito", "quito_2": "Quito", "guayaquil": "Guayaquil", "cuenca": "Cuenca"}
sedes_antes = sorted(df_curado["sede"].unique())
df_curado["sede"] = df_curado["sede"].str.strip().str.lower().map(MAPA_SEDES)
assert df_curado["sede"].notna().all(), "Hay una sede sin entrada en MAPA_SEDES"
bitacora.append(("Categorías inconsistentes de 'sede' unificadas", len(sedes_antes) - df_curado["sede"].nunique()))

df_curado["modalidad"] = df_curado["modalidad"].str.strip().str.title()

# ------------------------------------------------------------ b) Tipos de dato
df_curado["fecha_ultimo_acceso"] = pd.to_datetime(df_curado["fecha_ultimo_acceso"],
                                                  format="%Y/%m/%d", errors="coerce")
assert df_curado["fecha_ultimo_acceso"].notna().all(), "Alguna fecha no pudo parsearse"

fuera_ventana = int((df_curado["fecha_ultimo_acceso"] > FECHA_CORTE).sum())
df_curado["fecha_ultimo_acceso"] = df_curado["fecha_ultimo_acceso"].clip(upper=FECHA_CORTE)
bitacora.append((f"Fechas de último acceso censuradas a la semana {SEMANA_CORTE}", fuera_ventana))

df_curado["dias_hasta_ultimo_acceso"] = (df_curado["fecha_ultimo_acceso"] - FECHA_INICIO).dt.days.astype("int16")

for col in ["sede", "modalidad"]:
    df_curado[col] = df_curado[col].astype("category")
df_curado["entregas_semana_1_8"] = pd.to_numeric(df_curado["entregas_semana_1_8"], errors="coerce").astype("int16")
df_curado["n_registros"] = df_curado["n_registros"].astype("int8")

# ------------------------------------------------------------ c) Nulos
nulos_sat = int(df_curado["nivel_satisfaccion"].isna().sum())
moda_sat  = df_curado["nivel_satisfaccion"].mode(dropna=True).iloc[0]
df_curado["nivel_satisfaccion"] = df_curado["nivel_satisfaccion"].fillna(moda_sat).astype("int8")
bitacora.append((f"Nulos en 'nivel_satisfaccion' imputados con la moda ({moda_sat:.0f})", nulos_sat))

residuales = df_curado.isna().sum()
if residuales.sum():
    df_curado = df_curado.fillna(
        {c: (0 if pd.api.types.is_numeric_dtype(df_curado[c]) else df_curado[c].mode().iloc[0])
         for c in residuales[residuales > 0].index})
    bitacora.append(("Nulos residuales imputados (0 / moda)", int(residuales.sum())))

# ------------------------------------------------------------ d) Outliers: winsorización por IQR
def winsorizar_iqr(serie, factor=1.5, minimo_teorico=0.0):
    """Acota la serie a [Q1-1.5·IQR, Q3+1.5·IQR] sin eliminar filas. El piso nunca baja del mínimo teórico."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    li, ls = max(q1 - factor * iqr, minimo_teorico), q3 + factor * iqr
    n_out = int(((serie < li) | (serie > ls)).sum())
    return serie.clip(li, ls), n_out, li, ls

for col in ["tiempo_conexion_min"]:
    asim_antes = df_curado[col].skew()
    df_curado[col], n_out, li, ls = winsorizar_iqr(df_curado[col])
    bitacora.append((f"Outliers winsorizados en '{col}' (límites {li:.1f} / {ls:.1f})", n_out))
    print(f"Asimetría de '{col}': {asim_antes:.2f} -> {df_curado[col].skew():.2f} tras winsorizar\n")

# ------------------------------------------------------------ Bitácora y verificación
print(f"{'Acción de curaduría':<62} | {'Filas/valores':>13}")
print("-" * 80)
for accion, cantidad in bitacora:
    print(f"{accion:<62} | {cantidad:>13,}")

print(f"\n{'Métrica':<30} | {'Antes':>16} | {'Después':>16}")
print("-" * 70)
print(f"{'Dimensiones':<30} | {str(df_semestre.shape):>16} | {str(df_curado.shape):>16}")
print(f"{'Celdas nulas':<30} | {int(df_semestre.isna().sum().sum()):>16} | {int(df_curado.isna().sum().sum()):>16}")
print(f"{'Categorías de sede':<30} | {len(sedes_antes):>16} | {df_curado['sede'].nunique():>16}")
print(f"\nsede: {sedes_antes}  ->  {sorted(df_curado['sede'].cat.categories)}")
print(f"fecha_ultimo_acceso: {df_curado['fecha_ultimo_acceso'].dtype} "
      f"[{df_curado['fecha_ultimo_acceso'].min().date()} .. {df_curado['fecha_ultimo_acceso'].max().date()}]")

assert df_curado.isna().sum().sum() == 0
assert sorted(df_curado["sede"].cat.categories) == ["Cuenca", "Guayaquil", "Quito"]
assert df_curado["fecha_ultimo_acceso"].dtype.kind == "M"
assert (df_curado["fecha_ultimo_acceso"] <= FECHA_CORTE).all()
print("\nOK: tabla sin nulos, categorías normalizadas, tipos correctos y fechas dentro de la ventana.")

## 2.3 Filtro explícito por ventana temporal: eliminación automática del *data leakage*

**El problema.** El modelo debe decidir en la semana 8, cuando la calificación final todavía no existe. Si esa
columna entra como predictor, la validación arrojará métricas excelentes y el modelo será **inútil en
producción**: habrá aprendido a leer el resultado en lugar de anticiparlo.

**La solución.** `detectar_leakage` aplica tres reglas acumulativas, idénticas a las del análisis de
referencia, y documenta cuál se activó en cada caso:

| Regla | Criterio | Qué inspecciona |
|---|---|---|
| **R1 — temporal explícita** | el nombre referencia una semana `> 8` (regex, tomando el máximo de un rango) | el nombre |
| **R2 — semántica** | el nombre contiene un token de resultado o cierre | el nombre |
| **R3 — evidencia en los datos** | columna de tipo fecha cuyo **mínimo** ya supera la fecha de corte | el contenido |

Aquí aparece una diferencia importante respecto de un dataset con detalle semanal: **este dataset contiene una
sola variable futura**, así que se espera que solo R2 se active. Que R1 y R3 no encuentren nada no significa
que sobren — significa que este dataset no tiene el tipo de problema que ellas detectan. Para demostrar que
funcionan igualmente, la celda incluye una **prueba unitaria del detector** sobre una tabla ficticia de tres
columnas, construida solo para el test y descartada acto seguido: no entra en el análisis.

El objetivo se extrae **antes** del `drop`. Se derivan dos: `y_nota` (continuo, la calificación) y
`y_riesgo` (binario, `nota < 7`). `y_riesgo` no añade información — es una transformación de una columna que ya
existe— y por eso es igual de futura y queda fuera de los predictores.

In [ ]:
PATRON_SEMANA  = re.compile(r"sem(?:ana)?_?(\d+)(?:_(\d+))?")
TOKENS_FUTUROS = ["final", "cierre", "aprobado", "desercion", "riesgo",
                  "siguiente", "posterior", "total_semestre", "parcial_2"]

def semanas_en_columna(nombre):
    """Todos los números de semana referenciados en el nombre. Ej.: 'quiz_sem_9_16' -> [9, 16]."""
    numeros = []
    for g1, g2 in PATRON_SEMANA.findall(nombre):
        numeros.append(int(g1))
        if g2:
            numeros.append(int(g2))
    return numeros

def detectar_leakage(df, semana_corte=SEMANA_CORTE, fecha_corte=FECHA_CORTE):
    """Detecta columnas con información posterior a la ventana de observación.

    R1: el nombre referencia una semana mayor al corte.
    R2: el nombre contiene un token semántico de resultado/cierre.
    R3: es una columna de fecha cuyo valor mínimo ya supera la fecha de corte.
    """
    motivos = {}
    for col in df.columns:
        razones = []
        semanas = semanas_en_columna(col)
        if semanas and max(semanas) > semana_corte:
            razones.append(f"R1: referencia la semana {max(semanas)} > {semana_corte}")
        tokens = [t for t in TOKENS_FUTUROS if t in col.lower()]
        if tokens:
            razones.append(f"R2: token de resultado futuro {tokens}")
        if df[col].dtype.kind == "M" and df[col].min() > fecha_corte:
            razones.append(f"R3: todas las fechas son posteriores a {fecha_corte.date()}")
        if razones:
            motivos[col] = " | ".join(razones)
    return list(motivos), motivos

# ------------------------------------------------------------ Prueba unitaria del detector
_prueba = pd.DataFrame({
    "entregas_semana_1_8":    [1, 2],                                   # dentro de ventana -> debe sobrevivir
    "nota_parcial_2_sem12":   [7.0, 8.0],                               # R1 y R2
    "fecha_cierre_semestre":  pd.to_datetime(["2026-06-01", "2026-06-01"]),  # R2 y R3
})
_cols, _mot = detectar_leakage(_prueba)
assert set(_cols) == {"nota_parcial_2_sem12", "fecha_cierre_semestre"}, "El detector falla la prueba"
assert "R1" in _mot["nota_parcial_2_sem12"] and "R3" in _mot["fecha_cierre_semestre"]
print("Prueba unitaria del detector (sobre una tabla ficticia, no forma parte del análisis):")
for c, m in _mot.items():
    print(f"  detectada  {c:<24} | {m}")
print(f"  conservada entregas_semana_1_8   | referencia la semana 8, dentro de la ventana")
del _prueba, _cols, _mot

# ------------------------------------------------------------ Aplicación al dataset real
df_unidad = df_curado.set_index(CLAVE).sort_index()
cols_leakage, motivos = detectar_leakage(df_unidad)

print(f"\nVentana permitida: semanas 1 a {SEMANA_CORTE} (hasta {FECHA_CORTE.date()})")
print(f"Columnas auditadas: {df_unidad.shape[1]}  |  detectadas como fuga futura: {len(cols_leakage)}\n")
print(f"{'Columna eliminada':<32} | Motivo")
print("-" * 92)
for col in cols_leakage:
    print(f"{col:<32} | {motivos[col]}")
print(f"\nReglas que no se activaron en este dataset: "
      f"{[r for r in ['R1', 'R3'] if not any(r in m for m in motivos.values())]} "
      f"-> no hay columnas con detalle semanal posterior ni fechas de cierre.")

# ------------------------------------------------------------ Objetivos, extraídos antes del drop
y_nota   = df_unidad["calificacion_final_semestre"].rename("calificacion_final_semestre")
y_riesgo = (y_nota < 7).astype("int8").rename("riesgo_academico")

df_features = df_unidad.drop(columns=cols_leakage)

print(f"\nMatriz de predictores legítimos X : {df_features.shape[0]} filas x {df_features.shape[1]} columnas")
print(f"Objetivo continuo 'calificacion_final_semestre' : media = {y_nota.mean():.2f} | sd = {y_nota.std():.2f}")
print(f"Objetivo binario  'riesgo_academico' (nota < 7) : prevalencia = {y_riesgo.mean():.1%}")
print(f"\n{'Predictor conservado':<28} | Tipo")
print("-" * 50)
for c in df_features.columns:
    print(f"{c:<28} | {df_features[c].dtype}")

# ------------------------------------------------------------ Control de doble pasada
sobrevivientes, _ = detectar_leakage(df_features)
assert sobrevivientes == [], f"Aún hay columnas con leakage: {sobrevivientes}"
assert "calificacion_final_semestre" not in df_features.columns
print(f"\nOK: ninguno de los {df_features.shape[1]} predictores sobrevivientes viola la ventana temporal.")

df_viz = df_features.join(y_riesgo).join(y_nota)

### Lectura del resultado

Se eliminó **una sola columna**, `calificacion_final_semestre`, capturada por R2. Es el resultado correcto: el
dataset original contiene exactamente una variable futura. La prueba unitaria confirma que R1 y R3 funcionan;
simplemente no tienen nada que detectar aquí.

Merece atención `entregas_semana_1_8`, que **sobrevive** el filtro porque su nombre referencia la semana 8,
dentro de la ventana. La regla R1 hace aquí su trabajo en sentido positivo: no basta con eliminar todo lo que
suene temporal, hay que distinguir lo que ocurrió antes del corte de lo que ocurrió después.

Un caso más delicado es `tiempo_conexion_min`. El nombre no indica período alguno, así que **ninguna regla
automática la detecta**. Si esos minutos fueran los de todo el semestre, sería fuga de información; si son los
acumulados hasta la semana 8, es un predictor legítimo. El dataset no lo aclara, y ningún detector puede
resolverlo: **es una pregunta para quien produce los datos, no para el código**. Aquí se asume lo segundo, y
queda escrito como supuesto para que la decisión sea auditable.

---
# 3. Clasificación de variables y estrategia de *encoding*

## 3.1 Taxonomía

El tipo de una variable determina qué transformación es admisible: aplicar la codificación equivocada
**inventa una estructura que los datos no tienen**. Los dos errores clásicos son tratar una nominal como
numérica (imponer que Cuenca < Guayaquil < Quito) y tratar una ordinal como nominal (destruir el orden
1 < 2 < 3 < 4 < 5, que sí es información).

| # | Variable | Tipo | Escala | *Encoding* aplicado | Por qué |
|---|---|---|---|---|---|
| 1 | `entregas_semana_1_8` | **Discreta** (conteo) | Razón | ninguno | ya es un entero con cero absoluto y distancias interpretables |
| 2 | `dias_hasta_ultimo_acceso` | **Discreta** (días) | Razón | ninguno | derivada de la fecha censurada; convierte un `datetime` en magnitud usable |
| 3 | `tiempo_conexion_min` | **Continua** | Razón | `log1p` + estandarización *z* | asimetría derecha por la cola exponencial; el log la comprime y `log1p` admite el cero |
| 4 | `promedio_acumulado` | **Continua** | Intervalo | estandarización *z* | escala 5–10 sin cero absoluto; se centra para hacerla comparable |
| 5 | `nivel_satisfaccion` | **Ordinal** (1–5) | Ordinal | código ordinal (preserva el orden) | categorías ordenadas con distancias no necesariamente iguales; *one-hot* borraría el orden |
| 6 | `franja_entregas` | **Ordinal** (derivada) | Ordinal | *ordinal encoding* con orden explícito | discretización de `entregas_semana_1_8` en cuartiles: robustece frente a extremos y admite efectos no lineales |
| 7 | `modalidad` | **Binaria** | Nominal | 0/1 (`map`) | con dos categorías, un solo indicador basta y no genera colinealidad |
| 8 | `sede` | **Nominal politómica** | Nominal | *one-hot* con `drop_first` | sin orden posible; `drop_first` evita la trampa de las variables *dummy* (colinealidad perfecta) |

`franja_entregas` se construye con `pd.qcut` sobre el **rango** de `entregas_semana_1_8`, lo que garantiza
bordes únicos y cuartiles balanceados. El precio es que los empates —abundantes, al ser un conteo entero— se
reparten entre franjas contiguas, de modo que el máximo de una franja puede coincidir con el mínimo de la
siguiente. La estandarización usa `sklearn.StandardScaler` si está disponible y, si no, una implementación
equivalente en NumPy: el notebook no depende de instalar nada.

In [ ]:
CLASIFICACION_VARIABLES = [
    ("entregas_semana_1_8",      "Discreta (conteo)",  "Razón",     "Ninguno (ya numérica)"),
    ("dias_hasta_ultimo_acceso", "Discreta (días)",    "Razón",     "Ninguno (ya numérica)"),
    ("tiempo_conexion_min",      "Continua",           "Razón",     "log1p + estandarización z"),
    ("promedio_acumulado",       "Continua",           "Intervalo", "Estandarización z"),
    ("nivel_satisfaccion",       "Ordinal (1-5)",      "Ordinal",   "Código ordinal (preserva el orden)"),
    ("franja_entregas",          "Ordinal (derivada)", "Ordinal",   "Ordinal encoding con orden explícito"),
    ("modalidad",                "Binaria",            "Nominal",   "Binario 0/1 (map)"),
    ("sede",                     "Nominal politómica", "Nominal",   "One-hot con drop_first"),
]

print(f"{'#':>2} | {'Variable':<26} | {'Tipo':<20} | {'Escala':<10} | Encoding")
print("-" * 108)
for i, (v, t, e, enc) in enumerate(CLASIFICACION_VARIABLES, 1):
    print(f"{i:>2} | {v:<26} | {t:<20} | {e:<10} | {enc}")

df_enc = df_features.copy()

# ------------------------------------------------------- (6) Ordinal derivada: cuartiles de entregas
ORDEN_FRANJAS = ["Muy baja", "Baja", "Media", "Alta"]
df_enc["franja_entregas"] = pd.qcut(df_enc["entregas_semana_1_8"].rank(method="first"),
                                    q=4, labels=ORDEN_FRANJAS)
cortes = (df_enc.groupby("franja_entregas", observed=True)["entregas_semana_1_8"]
                .agg(entregas_min="min", entregas_max="max", n="count"))
print(f"\nDiscretización ordinal de 'entregas_semana_1_8' en cuartiles:\n{cortes.to_string()}")

# ------------------------------------------------------- (5)(6) Ordinales -> códigos que preservan el orden
df_enc["franja_entregas_ord"] = pd.Categorical(
    df_enc["franja_entregas"], categories=ORDEN_FRANJAS, ordered=True).codes.astype("int8")
df_enc["nivel_satisfaccion_ord"] = pd.Categorical(
    df_enc["nivel_satisfaccion"], categories=[1, 2, 3, 4, 5], ordered=True).codes.astype("int8")

# ------------------------------------------------------- (7) Binaria -> 0/1
MAPA_MODALIDAD = {"Presencial": 1, "En Línea": 0, "En Linea": 0}
df_enc["modalidad_presencial"] = df_enc["modalidad"].astype(str).map(MAPA_MODALIDAD)
assert df_enc["modalidad_presencial"].notna().all(), "Hay una modalidad sin entrada en MAPA_MODALIDAD"
df_enc["modalidad_presencial"] = df_enc["modalidad_presencial"].astype("int8")

# ------------------------------------------------------- (8) Nominal politómica -> one-hot con drop_first
df_enc = pd.concat([df_enc, pd.get_dummies(df_enc["sede"], prefix="sede",
                                           drop_first=True, dtype="int8")], axis=1)

# ------------------------------------------------------- (3)(4) Continuas -> log1p + estandarización z
try:
    from sklearn.preprocessing import StandardScaler
    escalar = lambda M: StandardScaler().fit_transform(M)
    print("\nEstandarización: sklearn.preprocessing.StandardScaler")
except ModuleNotFoundError:
    def escalar(M):
        """Equivalente a StandardScaler: z = (x - mu) / sigma, con sigma poblacional."""
        M  = np.asarray(M, dtype=float)
        sd = M.std(axis=0, ddof=0)
        return (M - M.mean(axis=0)) / np.where(sd == 0, 1.0, sd)
    print("\nEstandarización: implementación equivalente en NumPy (sklearn no disponible)")

df_enc["tiempo_conexion_log"] = np.log1p(df_enc["tiempo_conexion_min"])
CONTINUAS = ["tiempo_conexion_log", "promedio_acumulado"]
df_enc[[c + "_z" for c in CONTINUAS]] = escalar(df_enc[CONTINUAS].to_numpy(dtype=float))

# ------------------------------------------------------- Matriz de diseño final
YA_CODIFICADAS = ["sede", "modalidad", "franja_entregas", "nivel_satisfaccion", "fecha_ultimo_acceso"]
df_encoded = (df_enc
              .drop(columns=[c for c in YA_CODIFICADAS if c in df_enc.columns])
              .drop(columns=[c for c in CONTINUAS + ["tiempo_conexion_min"] if c in df_enc.columns]))

print(f"\nAsimetría de 'tiempo_conexion_min': {df_features['tiempo_conexion_min'].skew():.2f} "
      f"-> tras log1p: {df_enc['tiempo_conexion_log'].skew():.2f}")
print(f"\nDimensiones  antes del encoding : {df_features.shape[0]} x {df_features.shape[1]}")
print(f"Dimensiones después del encoding : {df_encoded.shape[0]} x {df_encoded.shape[1]}")
print(f"\n{'Columna de la matriz de diseño':<30} | {'Tipo':<10} | {'Media':>9} | {'sd':>9}")
print("-" * 68)
for c in df_encoded.columns:
    print(f"{c:<30} | {str(df_encoded[c].dtype):<10} | {df_encoded[c].mean():>9.3f} | {df_encoded[c].std():>9.3f}")

assert df_encoded.select_dtypes(include=["object", "category", "str"]).empty, "Quedan columnas no numéricas"
assert df_encoded.isna().sum().sum() == 0, "Quedan nulos en la matriz de diseño"
print("\nOK: matriz de diseño íntegramente numérica y sin nulos.")

### Lectura del resultado

La transformación logarítmica reduce la asimetría de `tiempo_conexion_min`, que es lo que evita que las
técnicas basadas en distancias o en supuestos de normalidad queden dominadas por la cola derecha. Nótese
también qué **no** se hizo: `sede` produce sólo dos columnas *dummy* para tres categorías, y la omitida
(Cuenca) queda como nivel de referencia — sin `drop_first`, las tres sumarían exactamente 1 y la matriz de
diseño sería singular.

---
# 4. Visualización exploratoria

Cinco gráficos con un propósito declarado cada uno: tres **distribuciones estratificadas** (G1–G3), un
diagnóstico de **sesgo de representación** (G4) y la **matriz de correlación** (G5). Todos usan `df_viz`, la
única vista donde predictores y objetivos coexisten, y exclusivamente para inspección visual.

In [ ]:
PALETA_SEDE  = {"Quito": "#4C72B0", "Guayaquil": "#DD8452", "Cuenca": "#55A868"}
COLOR_RIESGO = {"Sin riesgo": "#55A868", "En riesgo": "#C44E52"}

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(data=df_viz, x="tiempo_conexion_min", hue="sede", stat="density", common_norm=False,
             kde=True, bins=22, alpha=0.35, palette=PALETA_SEDE, ax=ax)
ax.set_title("G1. Distribución del tiempo de conexión estratificada por sede",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Minutos de conexión (winsorizados)")
ax.set_ylabel("Densidad")
plt.tight_layout()
plt.show()

print(df_viz.groupby("sede", observed=True)["tiempo_conexion_min"]
            .agg(n="count", media="mean", mediana="median", sd="std", asimetria="skew").round(1).to_string())

**G1 — Misma forma, distinta escala.** Las tres sedes muestran la firma característica de una distribución
exponencial: moda cerca de cero, cola larga a la derecha y asimetría positiva en todas (0.5 a 1.1). Las medias
difieren —Cuenca 158 min, Guayaquil 138, Quito 96— pero la **forma** es la misma, y las desviaciones estándar
acompañan a las medias en lugar de variar de forma independiente. Eso es lo esperable de una distribución
exponencial, donde media y desviación coinciden teóricamente: no indica sedes con comportamientos distintos,
sino tres muestras de la misma distribución.

Conviene no sobreinterpretar la diferencia entre 158 y 96 minutos: Cuenca tiene 24 estudiantes y Guayaquil 22.
Con muestras así de pequeñas y una cola tan larga, la media es un estadístico inestable — la mediana (115, 95 y
85) muestra diferencias bastante más discretas.

El corte abrupto en el extremo derecho es el techo de la winsorización: los 5 valores por encima de
Q3 + 1.5·IQR se acumularon en 365.7 minutos.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df_viz, x="sede", y="promedio_acumulado", hue="modalidad",
            palette="Set2", width=0.65, ax=ax)
sns.stripplot(data=df_viz, x="sede", y="promedio_acumulado", hue="modalidad",
              dodge=True, size=3.5, alpha=0.5, palette=["#333333", "#333333"], legend=False, ax=ax)
ax.set_title("G2. Promedio acumulado por sede y modalidad", fontsize=12, fontweight="bold")
ax.set_xlabel("Sede"); ax.set_ylabel("Promedio acumulado (escala 5-10)")
ax.legend(title="Modalidad", loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
plt.tight_layout()
plt.show()

print(df_viz.groupby(["sede", "modalidad"], observed=True)["promedio_acumulado"]
            .agg(n="count", media="mean", sd="std", rango_iqr=lambda s: s.quantile(.75) - s.quantile(.25))
            .round(2).to_string())

**G2 — Ni la sede ni la modalidad discriminan el promedio.** Las seis cajas se superponen casi por completo:
las medias van de 7.22 a 7.95, un rango de 0.73 puntos frente a desviaciones estándar de 1.19 a 1.85. La
diferencia entre grupos es **menor que la variabilidad dentro de cada grupo**, que es la definición operativa
de "no hay efecto".

Los IQR (de 2.06 a 3.19) tampoco sugieren dispersiones marcadamente distintas. La Sección 5 lo contrastará
formalmente en lugar de dejarlo en impresión visual.

Nótese el subgrupo Cuenca–En Línea: apenas 8 estudiantes. Una caja construida con 8 observaciones tiene
cuartiles muy inestables, y su aspecto no debe leerse al mismo nivel que la de Quito–Presencial, que tiene 27.

In [ ]:
df_g3 = df_viz.assign(estado=np.where(df_viz["riesgo_academico"] == 1, "En riesgo", "Sin riesgo"))
ORDEN_ESTADO = ["Sin riesgo", "En riesgo"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, var, etiqueta in [(axes[0], "entregas_semana_1_8", "Entregas acumuladas (semanas 1-8)"),
                          (axes[1], "promedio_acumulado",  "Promedio acumulado")]:
    sns.violinplot(data=df_g3, x="estado", y=var, order=ORDEN_ESTADO, hue="estado",
                   hue_order=ORDEN_ESTADO, palette=COLOR_RIESGO, inner="quartile",
                   legend=False, cut=0, ax=ax)
    sns.stripplot(data=df_g3, x="estado", y=var, order=ORDEN_ESTADO,
                  color="k", size=3, alpha=0.4, ax=ax)
    ax.set_title(etiqueta, fontsize=11); ax.set_xlabel(""); ax.set_ylabel(etiqueta)

fig.suptitle("G3. Predictores legítimos estratificados por riesgo académico",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

VARS_G3 = ["entregas_semana_1_8", "promedio_acumulado", "tiempo_conexion_min", "dias_hasta_ultimo_acceso"]
print(df_g3.groupby("estado", observed=True)[VARS_G3].mean().round(2).to_string())
print("\nSeparación estandarizada (d de Cohen) entre grupos:")
for v in VARS_G3:
    a = df_g3.loc[df_g3["estado"] == "Sin riesgo", v]
    b = df_g3.loc[df_g3["estado"] == "En riesgo",  v]
    s = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    d = (a.mean() - b.mean()) / s
    magnitud = "grande" if abs(d) >= 0.8 else "mediano" if abs(d) >= 0.5 else "pequeño" if abs(d) >= 0.2 else "DESPRECIABLE"
    print(f"  {v:<26} | d = {d:+.2f}  ({magnitud})")

**G3 — Los predictores no separan los grupos. Este es el hallazgo central del análisis.**

Los violines de estudiantes en riesgo y sin riesgo son prácticamente indistinguibles, y las *d* de Cohen lo
confirman: −0.19 en entregas, +0.16 en promedio acumulado, +0.07 en tiempo de conexión y −0.27 en días hasta
el último acceso. Todas por debajo del umbral de 0.2 que se considera un efecto pequeño; tres de las cuatro son
directamente despreciables.

Y hay algo peor que la magnitud: **el signo va en contra de la teoría**. Los estudiantes *en riesgo* entregan
en promedio más tareas (4.97 frente a 4.33) que los que no lo están. Si esto se tomara en serio, la conclusión
sería que entregar más tareas aumenta el riesgo de reprobar. Lo que en realidad indica es que la diferencia es
ruido muestral: cuando no hay señal, el signo de la diferencia es esencialmente aleatorio.

Este resultado no es un fallo del procedimiento. Es la respuesta correcta para estos datos, y saber leerla es
tan importante como saber detectar un patrón. Con estas variables y esta muestra, **no se puede predecir el
riesgo académico**. Un modelo entrenado aquí no superaría a la regla trivial de predecir siempre la clase
mayoritaria.

In [ ]:
tabla     = pd.crosstab([df_viz["sede"], df_viz["modalidad"]], df_viz["riesgo_academico"])
n_grupo   = tabla.sum(axis=1)
tasa      = tabla.div(n_grupo, axis=0)
etiquetas = [f"{s}\n{m}" for s, m in tasa.index]

fig, ax = plt.subplots(figsize=(11.5, 5))
abajo = np.zeros(len(tasa))
for clase, nombre in [(1, "En riesgo"), (0, "Sin riesgo")]:
    valores = tasa[clase].to_numpy() if clase in tasa.columns else np.zeros(len(tasa))
    ax.bar(etiquetas, valores, bottom=abajo, color=COLOR_RIESGO[nombre], label=nombre, width=0.7)
    abajo = abajo + valores

tasa_global    = df_viz["riesgo_academico"].mean()
umbral_pequeno = 0.6 * n_grupo.mean()
ax.axhline(tasa_global, color="k", ls="--", lw=1.2)
ax.text(len(tasa) - 0.45, tasa_global + 0.02, f"tasa global de riesgo: {tasa_global:.0%}", fontsize=9)
for i, (etq, nn) in enumerate(zip(etiquetas, n_grupo)):
    pequeno = nn < umbral_pequeno
    ax.text(i, 1.03, f"n={nn}", ha="center", fontsize=9,
            fontweight="bold" if pequeno else "normal", color="#C44E52" if pequeno else "k")
    if 1 in tasa.columns:
        ax.text(i, tasa[1].iloc[i] / 2, f"{tasa[1].iloc[i]:.0%}", ha="center", va="center",
                color="w", fontweight="bold")

ax.set_ylim(0, 1.12); ax.set_ylabel("Proporción de estudiantes")
ax.set_title("G4. Tasa de riesgo académico por sede × modalidad y tamaño de cada subgrupo",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
plt.tight_layout()
plt.show()

comparacion = pd.DataFrame({"n": n_grupo,
                            "tasa_riesgo": tasa[1] if 1 in tasa.columns else 0.0,
                            "desvio_vs_global": (tasa[1] if 1 in tasa.columns else 0.0) - tasa_global})
print(comparacion.round(3).to_string())
print(f"\nSubgrupo más pequeño : n = {n_grupo.min()}  ({n_grupo.idxmin()})")
print(f"Subgrupo más grande  : n = {n_grupo.max()}  ({n_grupo.idxmax()})")
print(f"Razón de tamaños     : {n_grupo.max() / n_grupo.min():.1f} a 1")
# Error estándar de una proporción: +-1.96*sqrt(p(1-p)/n) en el subgrupo más pequeño
p, nmin = tasa_global, n_grupo.min()
print(f"Margen de error (95%) de una tasa estimada con n={nmin}: ±{1.96 * np.sqrt(p * (1 - p) / nmin):.1%}")

**G4 — Un caso de manual de sesgo de representación.** Las tasas de riesgo oscilan entre el 57 % y el **100 %**,
lo que invita a concluir que Guayaquil–Presencial es un grupo en crisis. Los tamaños desmienten esa lectura:
ese subgrupo tiene **10 estudiantes**, y el más pequeño (Cuenca–En Línea) apenas 8.

El cálculo del margen de error lo cierra: con n = 8, el intervalo de confianza al 95 % de una proporción es de
**±31 puntos porcentuales**. Es decir, una tasa observada del 87.5 % es compatible con una tasa real de
cualquier valor entre el 56 % y el 100 %. Las diferencias entre subgrupos son, en su totalidad, más pequeñas
que la incertidumbre de las propias estimaciones.

El "100 % en riesgo" de Guayaquil–Presencial merece mención aparte, porque es el tipo de resultado que se cita
fuera de contexto: significa que 10 de 10 estudiantes tuvieron nota inferior a 7. Con una prevalencia global
del 71 %, la probabilidad de que eso ocurra por puro azar en un grupo de 10 es de 0.71¹⁰ ≈ 3 %. Poco probable
en aislamiento — pero se examinaron seis subgrupos, así que ver un caso extremo entre ellos es perfectamente
esperable.

Las tres consecuencias prácticas se mantienen aunque las diferencias no sean reales:

1. Las tasas de los subgrupos pequeños son **estadísticamente frágiles** y no deben leerse sin su intervalo.
2. Un modelo entrenado aquí optimizaría el error global y, por tanto, **el de los grupos grandes**; podría
   rendir peor en los pequeños sin que la métrica agregada lo revele.
3. La evaluación no puede ser sólo global: hay que **reportar métricas por sede y modalidad**.

El desbalance proviene de `np.random.choice` sobre cuatro categorías, una de las cuales (`quito_2`) se fusionó
con Quito y por eso Quito duplica al resto. En datos reales el mecanismo suele ser peor: cobertura desigual de
la instrumentación, sedes incorporadas al sistema en fechas distintas o registros perdidos de forma no
aleatoria.

In [ ]:
num  = df_encoded.join(y_riesgo).join(y_nota).select_dtypes(include=[np.number])
num  = num.loc[:, num.std(numeric_only=True) > 0]
corr = num.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            linewidths=.4, annot_kws={"size": 8}, cbar_kws={"shrink": .75, "label": "r de Pearson"}, ax=ax)
ax.set_title("G5. Matriz de correlación de Pearson (matriz de diseño + objetivos)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

from math import erf, sqrt, atanh

def p_correlacion(r, n):
    """p bilateral de un coeficiente de correlación, vía la transformación z de Fisher."""
    if abs(r) >= 1.0:
        return 0.0
    z = atanh(r) * sqrt(n - 3)
    return 2.0 * (1.0 - 0.5 * (1.0 + erf(abs(z) / sqrt(2.0))))

OBJETIVOS   = ["riesgo_academico", "calificacion_final_semestre"]
n_obs_corr  = len(num)
r_critico   = 1.96 / np.sqrt(n_obs_corr - 3)
predictores = [c for c in corr.columns if c not in OBJETIVOS]
N_COMPARACIONES = len(predictores) * len(OBJETIVOS)
ALFA_BONF   = 0.05 / N_COMPARACIONES

print(f"n = {n_obs_corr}  ->  |r| debe superar ~{r_critico:.3f} para ser distinguible de cero (alfa=0.05)")
print(f"Se evalúan {len(predictores)} predictores x {len(OBJETIVOS)} objetivos = {N_COMPARACIONES} correlaciones.")
print(f"Con {N_COMPARACIONES} pruebas a alfa=0.05 se esperan {0.05 * N_COMPARACIONES:.1f} falsos positivos por azar,")
print(f"así que el umbral corregido de Bonferroni es alfa = {ALFA_BONF:.4f}.\n")

resumen_corr = []
for objetivo in OBJETIVOS:
    ranking = corr[objetivo].drop(index=OBJETIVOS).sort_values(key=np.abs, ascending=False)
    print(f"Correlación con '{objetivo}':")
    for v, r in ranking.items():
        p = p_correlacion(r, n_obs_corr)
        if   p < ALFA_BONF: marca = "  ** significativa tras corregir por comparaciones múltiples"
        elif p < 0.05:      marca = "  *  significativa aislada, NO sobrevive la corrección"
        else:               marca = "     indistinguible de cero"
        print(f"  {v:<26} | r = {r:+.3f} | p = {p:.4f} |{marca}")
        resumen_corr.append({"objetivo": objetivo, "predictor": v, "r": r, "p": p})
    print()

df_corr = pd.DataFrame(resumen_corr)
print(f"{'Correlaciones con p < 0.05 (sin corregir)':<48}: "
      f"{(df_corr['p'] < 0.05).sum()} de {N_COMPARACIONES}  "
      f"(esperadas por azar si no hubiera relación: {0.05 * N_COMPARACIONES:.1f})")
print(f"{'Correlaciones que sobreviven Bonferroni':<48}: {(df_corr['p'] < ALFA_BONF).sum()} de {N_COMPARACIONES}")
print(f"{'|r| máximo observado':<48}: {df_corr['r'].abs().max():.3f}\n")

sup   = corr.where(mask.T & ~np.eye(len(corr), dtype=bool)).stack()
altos = sup[sup.abs() > 0.85]
altos = altos[[not ({a, b} & {"riesgo_academico", "calificacion_final_semestre"}) for a, b in altos.index]]
print("Pares de predictores con |r| > 0.85 (redundancia / multicolinealidad):")
print(altos.round(3).to_string() if len(altos) else "  (ninguno)")

**G5 — Ninguna correlación sobrevive el control de comparaciones múltiples.**

La matriz es casi uniformemente pálida. La correlación más fuerte con cualquiera de los dos objetivos es de
−0.292, y **ninguna de las 20 alcanza el umbral corregido de Bonferroni**.

El detalle importa, porque aquí se ve el error inferencial más común en análisis exploratorio. Dos
correlaciones superan el umbral individual de |r| > 0.205: `sede_Quito` con el riesgo (−0.292) y
`nivel_satisfaccion_ord` (+0.267). Tomadas de una en una, ambas serían "significativas al 5 %". Pero **no se
evaluó una correlación, se evaluaron 20**, y con 20 pruebas al 5 % se espera **un falso positivo por puro
azar**. Al corregir el umbral (α = 0.05/20 = 0.0025), ninguna sobrevive.

Sabemos con certeza que son falsos positivos porque conocemos el mecanismo generador: las columnas del dataset
se extraen con llamadas independientes a `np.random`, sin ninguna relación entre ellas. Es una situación
excepcional —en datos reales nunca se conoce la verdad— y por eso resulta tan instructiva: **el análisis
exploratorio produjo dos "hallazgos" significativos sobre datos donde, por construcción, no hay nada que
hallar**. Reportar `sede_Quito` como factor protector sería un artefacto del procedimiento, no un
descubrimiento.

Las dos correlaciones intensas del mapa son **definicionales, no empíricas**. `franja_entregas_ord` con
`entregas_semana_1_8` (r = 0.944) porque la primera es una discretización de la segunda; en un modelo lineal
habría que incluir sólo una de las dos. Y `riesgo_academico` con `calificacion_final_semestre` (r = −0.81)
porque el riesgo se definió como *nota < 7*: son el mismo objetivo en dos escalas, y por eso el filtro de
redundancia las excluye — sólo busca pares entre **predictores**.

Un último aviso que el mapa deja ver: entre predictores también hay asociaciones aparentes, como
`dias_hasta_ultimo_acceso` con `modalidad_presencial` (r = 0.36). Tampoco tiene mecanismo detrás. Al examinar
decenas de celdas de una matriz de correlación, **algunas se saldrán de lo esperado por pura combinatoria**, y
la única defensa es fijar de antemano qué hipótesis se van a contrastar en lugar de recorrer la matriz
buscando la casilla más intensa.

---
# 5. Análisis estadístico matricial con NumPy

## 5.1 Marco conceptual

Sea $X$ la matriz de datos de $n$ observaciones y $p$ indicadores, con $X_{ij}$ el valor del indicador $j$ en
la observación $i$.

**Esperanza.** Estimada por la media muestral, es el centro de la distribución de cada indicador:

$$\hat{\mathbb{E}}[X_j] = \bar{x}_j = \frac{1}{n}\sum_{i=1}^{n} X_{ij}$$

**Varianza.** Dispersión promedio al cuadrado respecto de la media. Se usa $n-1$ (`ddof=1`) porque estimar la
media con los mismos datos consume un grado de libertad; con $n$ el estimador sería sesgado a la baja:

$$\widehat{\operatorname{Var}}(X_j) = s_j^2 = \frac{1}{n-1}\sum_{i=1}^{n}\left(X_{ij}-\bar{x}_j\right)^2$$

**Covarianza.** Mide la variación conjunta: positiva si los indicadores tienden a desviarse de su media en el
mismo sentido. Su expresión matricial, con $X_c$ la matriz centrada por columnas, es:

$$S = \frac{1}{n-1}\,X_c^{\top} X_c \qquad\text{con}\qquad S_{jk}=\widehat{\operatorname{Cov}}(X_j,X_k)$$

La diagonal de $S$ contiene las varianzas. La covarianza **depende de las unidades** (minutos × puntos no es
interpretable), por lo que se normaliza a la matriz de correlación, adimensional y acotada en $[-1,1]$:

$$R = D^{-1} S D^{-1}, \qquad D = \operatorname{diag}(s_1,\dots,s_p)$$

**Heterocedasticidad.** Los métodos clásicos (mínimos cuadrados ordinarios, ANOVA, pruebas *t*) suponen
**homocedasticidad**: varianza constante entre grupos. Si las sedes tienen dispersiones distintas,

$$\operatorname{Var}(X \mid \text{sede}=g) \neq \sigma^2 \quad \text{para todo } g,$$

las estimaciones puntuales siguen siendo no sesgadas, pero **los errores estándar dejan de ser válidos**: los
intervalos de confianza y los *p*-valores quedan mal calibrados. Se contrasta con el test de **Levene** en su
variante **Brown–Forsythe** (centrado en la mediana, robusto a la no normalidad):

$$W=\frac{(N-k)\sum_{g} n_g(\bar{Z}_g-\bar{Z})^2}{(k-1)\sum_{g}\sum_{i}(Z_{ig}-\bar{Z}_g)^2},
\qquad Z_{ig}=\left|X_{ig}-\operatorname{med}(X_g)\right|$$

bajo $H_0$ de igualdad de varianzas, $W \sim F_{k-1,\,N-k}$.

> **Nota de implementación.** `scipy` no está disponible en este entorno, así que `levene_brown_forsythe` y el
> *p*-valor de la $F$ se implementan directamente. La cola de la $F$ se obtiene de la **función beta
> incompleta regularizada** mediante la fracción continua de *Numerical Recipes*:
> $P(F_{d_1,d_2} \ge f) = I_{x}\!\left(\tfrac{d_2}{2}, \tfrac{d_1}{2}\right)$ con
> $x = d_2/(d_2 + d_1 f)$. La implementación se **valida contra valores tabulados** y, si `scipy` estuviera
> instalado, se contrasta además de forma cruzada con `scipy.stats.levene`.

In [ ]:
from math import lgamma, log, exp

def _betacf(a, b, x, itmax=300, eps=3e-16):
    """Fracción continua de Lentz para la función beta incompleta (Numerical Recipes)."""
    TINY = 1e-30
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c, d = 1.0, 1.0 - qab * x / qap
    if abs(d) < TINY: d = TINY
    d = 1.0 / d
    h = d
    for m in range(1, itmax + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d;  d = TINY if abs(d) < TINY else d;  d = 1.0 / d
        c = 1.0 + aa / c;  c = TINY if abs(c) < TINY else c
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d;  d = TINY if abs(d) < TINY else d;  d = 1.0 / d
        c = 1.0 + aa / c;  c = TINY if abs(c) < TINY else c
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return h

def betainc_reg(a, b, x):
    """Función beta incompleta regularizada I_x(a, b)."""
    if x <= 0.0: return 0.0
    if x >= 1.0: return 1.0
    bt = exp(lgamma(a + b) - lgamma(a) - lgamma(b) + a * log(x) + b * log(1.0 - x))
    if x < (a + 1.0) / (a + b + 2.0):
        return bt * _betacf(a, b, x) / a
    return 1.0 - bt * _betacf(b, a, 1.0 - x) / b

def p_valor_F(F, gl1, gl2):
    """Cola superior de la distribución F: P(F_{gl1,gl2} >= F)."""
    if not np.isfinite(F) or F <= 0:
        return 1.0
    return betainc_reg(gl2 / 2.0, gl1 / 2.0, gl2 / (gl2 + gl1 * F))

def levene_brown_forsythe(*grupos):
    """Test de Levene centrado en la mediana (Brown-Forsythe). Devuelve (W, gl1, gl2, p)."""
    grupos = [np.asarray(g, dtype=float) for g in grupos if len(g) > 1]
    k      = len(grupos)
    z      = [np.abs(g - np.median(g)) for g in grupos]      # desvíos absolutos a la mediana
    n_tot  = sum(len(g) for g in grupos)
    z_bar  = [zi.mean() for zi in z]
    z_glob = np.concatenate(z).mean()
    entre  = sum(len(zi) * (zb - z_glob) ** 2 for zi, zb in zip(z, z_bar)) / (k - 1)
    dentro = sum(((zi - zb) ** 2).sum() for zi, zb in zip(z, z_bar)) / (n_tot - k)
    W = entre / dentro if dentro > 0 else np.inf
    return W, k - 1, n_tot - k, p_valor_F(W, k - 1, n_tot - k)

# ------------------------------------------------------------ Validación contra valores tabulados
REFERENCIAS = [                     # (F, gl1, gl2, p esperado)
    (4.1028, 2, 10, 0.05),          # F crítico al 5% con 2 y 10 gl
    (3.0984, 3, 20, 0.05),          # F crítico al 5% con 3 y 20 gl
    (7.5594, 2, 10, 0.01),          # F crítico al 1% con 2 y 10 gl
    (1.0000, 1,  1, 0.50),          # mediana de F(1,1)
]
assert abs(betainc_reg(2.0, 3.0, 0.5) - 0.6875) < 1e-9, "beta incompleta mal calculada"
print("Validación de la implementación contra valores tabulados:")
print(f"  I_0.5(2,3) = {betainc_reg(2.0, 3.0, 0.5):.6f}   (exacto: 0.687500)")
for F, gl1, gl2, esperado in REFERENCIAS:
    obtenido = p_valor_F(F, gl1, gl2)
    assert abs(obtenido - esperado) < 1e-4, f"cola de la F mal calibrada en F({gl1},{gl2})"
    print(f"  P(F_{gl1},{gl2} >= {F:7.4f}) = {obtenido:.6f}   (tabla: {esperado:.4f})")

try:
    from scipy import stats as _st
    SCIPY_OK = True
    print("\nscipy disponible: los resultados se validarán de forma cruzada con scipy.stats.levene.")
except ModuleNotFoundError:
    SCIPY_OK = False
    print("\nscipy no disponible: se usa exclusivamente la implementación propia en NumPy.")

## 5.2 Esperanza, varianza y matriz de covarianza de los indicadores

Se analizan los cinco indicadores numéricos disponibles, todos dentro de la ventana permitida. La matriz de
covarianza se calcula con `np.cov` y se **verifica contra la definición matricial** $X_c^\top X_c/(n-1)$
mediante `np.allclose`.

In [ ]:
INDICADORES = ["promedio_acumulado", "tiempo_conexion_min", "entregas_semana_1_8",
               "nivel_satisfaccion", "dias_hasta_ultimo_acceso"]

X = df_features[INDICADORES].to_numpy(dtype=float)
n_obs, p_var = X.shape

esperanza  = X.mean(axis=0)
varianza   = X.var(axis=0, ddof=1)
desviacion = np.sqrt(varianza)
S          = np.cov(X, rowvar=False, ddof=1)
R          = np.corrcoef(X, rowvar=False)

# Verificación contra la definición matricial
Xc = X - esperanza
S_manual = Xc.T @ Xc / (n_obs - 1)
assert np.allclose(S, S_manual),          "np.cov no coincide con Xc'Xc/(n-1)"
assert np.allclose(np.diag(S), varianza), "la diagonal de S no coincide con las varianzas"
D_inv = np.diag(1.0 / desviacion)
assert np.allclose(D_inv @ S @ D_inv, R), "R no coincide con D^-1 S D^-1"

print(f"Matriz de indicadores X: {n_obs} observaciones x {p_var} variables "
      f"(unidad: estudiante-semestre)\n")
print(f"{'Indicador':<26} | {'E[X]':>10} | {'Var(X)':>12} | {'sd':>9} | {'CV':>7} | {'Asimetría':>9}")
print("-" * 92)
for i, v in enumerate(INDICADORES):
    print(f"{v:<26} | {esperanza[i]:>10.3f} | {varianza[i]:>12.3f} | {desviacion[i]:>9.3f} | "
          f"{desviacion[i] / esperanza[i]:>6.1%} | {df_features[v].skew():>9.2f}")

print("\nMatriz de covarianza S = Xc'Xc/(n-1)  [verificada con np.allclose]:")
print(pd.DataFrame(S, index=INDICADORES, columns=INDICADORES).round(2).to_string())

print("\nMatriz de correlación R = D^-1 S D^-1:")
print(pd.DataFrame(R, index=INDICADORES, columns=INDICADORES).round(3).to_string())

print(f"\ntraza(S) = {np.trace(S):>14,.2f}   (varianza total)")
print(f"det(S)   = {np.linalg.det(S):>14.3e}   rango = {np.linalg.matrix_rank(S)} de {p_var}")

lam_S = np.linalg.eigvalsh(S)[::-1]
lam_R = np.linalg.eigvalsh(R)[::-1]
print(f"\nAutovalores de S : {np.array2string(lam_S, precision=2)}")
print(f"  el 1er componente concentra {lam_S[0] / lam_S.sum():.1%} de la traza, pero es un ARTEFACTO DE ESCALA:")
print(f"  '{INDICADORES[int(np.argmax(varianza))]}' aporta por sí solo {varianza.max() / varianza.sum():.1%} "
      f"de la varianza total por su unidad de medida.")
print(f"\nAutovalores de R : {np.array2string(lam_R, precision=3)}   (escala-invariante)")
print(f"  el 1er componente explica {lam_R[0] / p_var:.1%} de la varianza estandarizada")
print(f"  número de condición de R = {lam_R[0] / lam_R[-1]:.1f} -> "
      f"{'sin' if lam_R[0] / lam_R[-1] < 30 else 'CON'} multicolinealidad severa")
print(f"\nReferencia: si los indicadores fueran mutuamente independientes, R seria la identidad,")
print(f"todos sus autovalores valdrían 1.000 y el número de condición sería exactamente 1.0.")

### Lectura del resultado

**Los coeficientes de variación ordenan las variables por heterogeneidad.** `promedio_acumulado` tiene el CV
más bajo (19.0 %): es una medida acotada y homogénea entre estudiantes. `tiempo_conexion_min` alcanza el 84.6 %,
lo esperable de una distribución exponencial, donde la desviación estándar iguala teóricamente a la media
(aquí 103 frente a 122, algo por debajo por efecto de la winsorización).

**La covarianza en unidades originales es engañosa, y el análisis espectral lo demuestra.** El primer
autovalor de $S$ concentra el 96.9 % de la traza. Podría parecer que existe un factor latente dominante, pero
la cifra siguiente lo desmiente: `tiempo_conexion_min` aporta por sí sola el 96.9 % de la varianza total,
simplemente por estar medida en minutos y no en una escala de 0 a 10. **El "componente principal" es la unidad
de medida.** De ahí la regla práctica: cualquier técnica basada en la matriz de covarianza —PCA incluido—
requiere estandarizar cuando las variables no comparten unidad, que es exactamente la transformación aplicada
en la Sección 3.

**La matriz de correlación es casi la identidad, y eso es el resultado.** Los autovalores de $R$ son
1.328, 1.264, 0.912, 0.813 y 0.683: todos próximos a 1, que es el valor que tomarían si las cinco variables
fueran mutuamente independientes. El número de condición es 1.9, frente al 1.0 de la independencia perfecta.
Fuera de la diagonal, la correlación mayor en valor absoluto es 0.206.

Dicho de otro modo: **la estructura de covarianza de estos cinco indicadores es indistinguible de la de cinco
variables sin relación alguna**. No hay factores latentes, no hay redundancia y no hay multicolinealidad —
pero tampoco hay estructura que un modelo pueda aprovechar.

## 5.3 ¿Hay heterocedasticidad entre sedes?

Se contrasta, indicador por indicador, si la varianza es constante entre sedes, combinando dos criterios:

- **Razón de varianzas** $\max_g s_g^2 / \min_g s_g^2$, con el umbral práctico habitual de **4**
  (equivalente a un factor 2 en desviaciones estándar).
- **Test de Levene / Brown–Forsythe** con $\alpha = 0{,}05$.

Se declara heterocedasticidad cuando **cualquiera** de los dos criterios se activa: la razón detecta
diferencias grandes aunque la muestra sea pequeña, y el test detecta diferencias moderadas pero sistemáticas.
Se calculan además las **matrices de covarianza por sede** completas, porque la heterocedasticidad puede
afectar no sólo a las varianzas marginales sino a la estructura de asociación entre indicadores.

Conviene fijar de antemano la potencia disponible: con ~30 estudiantes por sede, el test solo detectará
diferencias de varianza **grandes**. Un resultado no significativo aquí significa "no hay evidencia de
diferencia grande", no "las varianzas son iguales".

In [ ]:
sedes        = list(df_features["sede"].cat.categories)
UMBRAL_RATIO = 4.0
ALFA         = 0.05

var_por_sede = pd.DataFrame(
    [[df_features.loc[df_features["sede"] == s, v].var(ddof=1) for v in INDICADORES] for s in sedes],
    index=sedes, columns=INDICADORES)
media_por_sede = pd.DataFrame(
    [[df_features.loc[df_features["sede"] == s, v].mean() for v in INDICADORES] for s in sedes],
    index=sedes, columns=INDICADORES)

print("Tamaño muestral por sede:")
print(df_features["sede"].value_counts().to_string())
print("\nMedia por sede:")
print(media_por_sede.round(2).to_string())
print("\nVarianza por sede (ddof=1):")
print(var_por_sede.round(2).to_string())

print(f"\n{'Indicador':<26} | {'ratio var':>9} | {'W (B-F)':>9} | {'gl':>8} | {'p-valor':>9} | Veredicto")
print("-" * 96)
filas_het = []
for v in INDICADORES:
    grupos = [df_features.loc[df_features["sede"] == s, v].to_numpy(dtype=float) for s in sedes]
    ratio  = var_por_sede[v].max() / var_por_sede[v].min()
    W, gl1, gl2, p = levene_brown_forsythe(*grupos)
    if SCIPY_OK:
        W_ref, p_ref = _st.levene(*grupos, center="median")
        assert abs(W - W_ref) < 1e-8 and abs(p - p_ref) < 1e-6, "discrepancia con scipy.stats.levene"
    hetero = (p < ALFA) or (ratio > UMBRAL_RATIO)
    print(f"{v:<26} | {ratio:>9.2f} | {W:>9.3f} | {gl1:>3},{gl2:<4} | {p:>9.4f} | "
          f"{'HETEROCEDÁSTICO' if hetero else 'homocedástico'}")
    filas_het.append({"indicador": v, "ratio_var": ratio, "W": W, "p_valor": p,
                      "heterocedastico": hetero,
                      "sede_max_var": var_por_sede[v].idxmax(),
                      "sede_min_var": var_por_sede[v].idxmin()})

df_het = pd.DataFrame(filas_het).set_index("indicador")
print(f"\nCriterio: p < {ALFA} (Levene/Brown-Forsythe) O razón de varianzas > {UMBRAL_RATIO}")
print(f"Indicadores con evidencia de heterocedasticidad entre sedes: "
      f"{int(df_het['heterocedastico'].sum())} de {len(df_het)}")
print(f"\n{df_het.round(4).to_string()}")

print("\n" + "=" * 96)
print("MATRICES DE COVARIANZA POR SEDE (la diagonal contiene las varianzas)")
print("=" * 96)
cov_por_sede = {}
for s in sedes:
    Xs = df_features.loc[df_features["sede"] == s, INDICADORES].to_numpy(dtype=float)
    Ss = np.cov(Xs, rowvar=False, ddof=1)
    cov_por_sede[s] = Ss
    print(f"\n--- {s}  (n = {Xs.shape[0]}, traza = {np.trace(Ss):,.1f}, det = {np.linalg.det(Ss):.2e}) ---")
    print(pd.DataFrame(Ss, index=INDICADORES, columns=INDICADORES).round(1).to_string())

traza_g = pd.Series({s: np.trace(M) for s, M in cov_por_sede.items()})
print(f"\nVarianza total (traza) por sede: {traza_g.round(1).to_dict()}")
print(f"Razón entre la traza mayor y la menor: {traza_g.max() / traza_g.min():.2f} a 1")

In [ ]:
sd_relativa = np.sqrt(var_por_sede).div(np.sqrt(varianza), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [1.35, 1]})

sd_relativa.T.plot(kind="bar", ax=axes[0], color=[PALETA_SEDE.get(s, "#888888") for s in sd_relativa.index],
                   edgecolor="white", width=0.8)
axes[0].axhline(1.0, color="k", ls="--", lw=1.2)
axes[0].text(-0.45, 1.03, "desviación global", fontsize=8)
axes[0].set_title("Desviación estándar por sede relativa a la global", fontsize=11)
axes[0].set_ylabel("sd de la sede / sd global"); axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_ylim(0, 1.60)
axes[0].legend(title="Sede", fontsize=8, ncol=3, loc="upper center", framealpha=0.9)

colores = ["#C44E52" if h else "#55A868" for h in df_het["heterocedastico"]]
axes[1].barh(df_het.index, df_het["ratio_var"], color=colores, edgecolor="white")
axes[1].axvline(UMBRAL_RATIO, color="k", ls="--", lw=1.2)
axes[1].text(UMBRAL_RATIO * 1.02, -0.45, f"umbral = {UMBRAL_RATIO:g}", fontsize=8)
for i, (r, p) in enumerate(zip(df_het["ratio_var"], df_het["p_valor"])):
    axes[1].text(r + 0.05, i, f"p={p:.3f}", va="center", fontsize=8)
axes[1].set_title("Razón de varianzas máx/mín entre sedes", fontsize=11)
axes[1].set_xlabel("razón de varianzas")
axes[1].set_xlim(0, max(df_het["ratio_var"].max() * 1.45, UMBRAL_RATIO * 1.25))

fig.suptitle("Diagnóstico de heterocedasticidad entre sedes", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Veredicto sobre la heterocedasticidad

**No hay evidencia de heterocedasticidad entre sedes: 0 de 5 indicadores.** Ninguna razón de varianzas se
acerca al umbral de 4 (van de 1.29 a 2.26) y ningún *p*-valor baja de 0.05; el más pequeño es 0.0825, en
`tiempo_conexion_min`. La traza de la matriz de covarianza varía 2.21 a 1 entre Cuenca y Quito, una diferencia
que arrastra casi por completo la escala de `tiempo_conexion_min` y que el test no considera significativa.

**El caso de `tiempo_conexion_min` (razón 2.26, *p* = 0.0825) merece una lectura cuidadosa.** Es tentador
describirlo como "marginalmente significativo" o "una tendencia a la heterocedasticidad". Sería incorrecto:
*p* = 0.0825 significa que una diferencia de varianzas al menos así de grande ocurre el 8 % de las veces
cuando las varianzas son idénticas. Y hay una razón estructural que la explica sin recurrir a diferencias
entre sedes: en una distribución exponencial la varianza es el **cuadrado** de la media, de modo que las medias
observadas (158, 138 y 96 minutos) generan varianzas esperadas en proporción 2.7 a 1 aunque las tres sedes
provengan de la misma población. La razón observada, 2.26, es menor que eso.

**Sobre la potencia del contraste.** Con 22 a 48 estudiantes por sede, este test sólo detecta diferencias de
varianza grandes. El resultado correcto no es "las varianzas son iguales" sino **"no hay evidencia de que
difieran"**, y esas dos afirmaciones no son intercambiables. Una diferencia moderada y real podría pasar
inadvertida con esta muestra.

**Implicación práctica.** Al no rechazarse la homocedasticidad, los métodos clásicos —MCO, ANOVA, pruebas
*t*— no requieren corrección por este motivo en particular. Es la única condición que estos datos cumplen sin
reparos, y conviene ponerla en perspectiva: cumplir el supuesto de varianza constante no aporta nada cuando no
hay ninguna relación que modelar. La homocedasticidad habilita las herramientas; no crea la señal.

---
# 6. Conclusiones

**1. La unidad de análisis la impusieron los datos.** Sin columna de asignatura, la unidad
estudiante-asignatura era inconstruible sin inventar información, así que la única unidad legítima es
**estudiante-semestre**. Verificar previamente con las fechas que el dataset cubre un único período (99 días)
fue lo que permitió afirmar que `id_estudiante` identifica la unidad. La agregación consolidó 100 registros en
94 estudiantes.

**2. Los duplicados eran un problema de ingesta, no de limpieza.** Los 6 identificadores repetidos no son
copias: son registros **conflictivos**, y en 4 casos con `sede` o `modalidad` contradictorias. Resolverlos por
agregación —con una regla declarada por columna— conserva la información que `drop_duplicates` habría
descartado, y deja constancia de que en esos 4 casos el criterio `first()` decidió de forma arbitraria.

**3. El control de fuga funcionó, aunque tuviera poco que hacer.** Se eliminó una sola columna,
`calificacion_final_semestre`. Que R1 y R3 no se activaran no invalida las reglas: la **prueba unitaria** sobre
una tabla ficticia demuestra que detectan lo que deben. Queda documentado el punto ciego real:
`tiempo_conexion_min` no declara período en su nombre, y **ninguna regla automática puede determinar** si son
los minutos hasta la semana 8 o los de todo el semestre. Eso se le pregunta a quien produce los datos.

**4. Las ocho variables se clasificaron y codificaron según su escala**, y el `drop_first` en `sede` evitó la
singularidad de la matriz de diseño.

**5. No hay señal predictiva. Es el hallazgo principal, y es un resultado, no una carencia.** Tres análisis
independientes convergen:

| Evidencia | Resultado |
|---|---|
| *d* de Cohen entre grupos de riesgo (G3) | de −0.27 a +0.16, todas despreciables; con **el signo invertido** respecto de la teoría |
| Correlaciones con los objetivos (G5) | máximo \|r\| = 0.292; **0 de 20 sobreviven la corrección de Bonferroni** |
| Autovalores de $R$ (§5.2) | 1.33 a 0.68, número de condición 1.9 — indistinguible de la identidad |

Un modelo entrenado sobre estos datos no superaría a predecir siempre la clase mayoritaria. Y las dos
correlaciones "significativas" de G5 ilustran el riesgo del análisis exploratorio sin control: con 20 pruebas
al 5 % se espera un falso positivo, y aquí aparecieron sobre datos que **por construcción no contienen ninguna
relación**.

**6. Los datos son homocedásticos entre sedes, y también aquí conviene precisión.** Cero de cinco indicadores
muestran evidencia (razones de 1.29 a 2.26, *p* mínimo 0.0825). Con 22–48 estudiantes por sede la conclusión
correcta es "no hay evidencia de diferencia", no "las varianzas son iguales". El caso de `tiempo_conexion_min`
tiene además una explicación estructural: en una distribución exponencial la varianza es el cuadrado de la
media, así que basta con que las medias difieran para que las varianzas lo hagan.

**7. El problema real es el tamaño muestral, y esto sí se traslada a datos reales.** Con 94 estudiantes y una
prevalencia del 71 %, la clase minoritaria tiene 27 casos. Aplicando la regla de ≥10 eventos por variable, el
presupuesto es de **2 predictores**, frente a los 9 disponibles. Los subgrupos de `sede × modalidad` bajan a 8
estudiantes, con un margen de error de ±31 puntos en cualquier tasa que se estime sobre ellos. Aunque existiera
señal, esta muestra sería demasiado pequeña para detectarla con fiabilidad.

**Qué se lleva uno de aquí.** El procedimiento —unidad de análisis declarada y verificada, curaduría con
bitácora, control de fuga con pruebas y aserciones, codificación por tipo, y diagnóstico de (co)variabilidad
con corrección por comparaciones múltiples— es válido y reutilizable. Aplicado a estos datos, su conclusión es
que **no hay nada que modelar**. Que el método sea capaz de decir "aquí no hay nada" en lugar de fabricar un
hallazgo es precisamente lo que lo hace confiable cuando sí lo haya.